# What does a softened teacher teach?

Day 26 · Chapter 15 · CPU worked notebook

Read Chapter15 sections15.5–15.7. Predict: does increasing T without T² scaling preserve the gradient magnitude? Can the teacher receive gradients in standard distillation?

Saved reference preview from the bounded CPU reference execution. It is a fixed result; run the cells to regenerate figures after changing controls. Make the prediction above before inspecting the preview.

![Saved reference 1: temperature distillation](../figures/chapter-15/day-26-01_temperature_distillation-01.png)

In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dongxi_llms").is_dir())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.figsize": (8, 3.8), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
# CPU only. Local original fixtures; no checkpoint/network/server needed.


In [ ]:
from dongxi_llms.distillation_lab import distillation_loss, distill_distribution
result = distill_distribution(temperature=2.)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
history = result["history"]
axes[0].semilogy([r["update"] for r in history], [r["scaled_loss"] for r in history])
axes[0].set(xlabel="Student SGD update", ylabel="T² forward KL")
for i in range(3):
    axes[1].plot([r["update"] for r in history], [r["student_p"][i] for r in history], label=f"Student outcome {i}")
    axes[1].axhline(history[-1]["teacher_p"][i], linestyle=":", color=f"C{i}")
axes[1].set(xlabel="Student SGD update", ylabel="Probability at inference T1")
axes[1].legend()
plt.show()

Reading guide: loss uses training temperature2; the displayed probabilities use inference temperature1. Dashed lines are the fixed teacher distribution.

Exercise: compare analytical gradients at T1,T2,T4, with and without the T² multiplier.

In [ ]:
teacher = torch.tensor([[2., .5, -1.]], dtype=torch.float64, requires_grad=True)
for T in (1., 2., 4.):
    for scale in (False, True):
        student = torch.zeros_like(teacher, requires_grad=True)
        loss = distillation_loss(student, teacher, T, scale=scale)
        loss.backward()
        residual = (student.detach()/T).softmax(-1)-(teacher.detach()/T).softmax(-1)
        analytical = residual*(T if scale else 1/T)
        torch.testing.assert_close(student.grad, analytical)
        print("T,scale,gradient:", T, scale, student.grad.tolist())
assert teacher.grad is None
print("Teacher detached; outcome coordinates must use one aligned vocabulary.")

Controlled change: reverse the teacher's favored outcome or initialize the student confidently wrong. Evidence boundary: a fitted three-logit distribution does not establish model compression, reasoning transfer, robustness or faster inference.